# Robustness Validation — Checkpoint ep398 (Sharpe 1.259)

**Purpose**: Comprehensive, publication-quality evaluation of the best Run 5 checkpoint.  
**Checkpoint**: `exp6_tape_hw_ep00398_shp1p259`  
**OOS Window**: 2020-01-02 → 2025-08-29  

### Notebook Structure
1. **Setup** — Colab sync, Drive mount, checkpoint restore
2. **Config & Dataset** — Metadata lock, normalized features, test split
3. **Full OOS Deterministic Evaluation** — Multiple start dates × horizons
4. **Multi-Horizon Stochastic Robustness (30 runs × 5 horizons)** — 1yr, 2yr, 3yr, 4yr, full
5. **Full OOS Single Pass** — Primary equity curve with daily weights/alphas
6. **Cross-Horizon Stochastic Analysis** — Robustness comparison across horizons
7. **Data Export & Drive Backup**

---
## 1) Connect to Colab VM and sync repository

In [1]:
import os, subprocess, sys

EVAL_REPO_DIR = "/content/tcn_tape_vectorized_version_clean"
EVAL_BRANCH = "feature/experimental-updates-20260302" # "main"
EVAL_GH_REPO = "Dave-DKings/tcn_tape_vectorized_version"

if not os.path.exists(EVAL_REPO_DIR):
    subprocess.run(["git", "clone", "-b", EVAL_BRANCH,
                    f"https://github.com/{EVAL_GH_REPO}.git", EVAL_REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", EVAL_REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(EVAL_REPO_DIR)
print("[OK] Repo synced")
print("   repo:", EVAL_REPO_DIR)
print("   branch:", EVAL_BRANCH)

In [2]:
RUN_ID = "run5"
CHECKPOINT_TAG = "exp6_tape_hw_ep00398_shp1p259"

In [3]:
# Quick GPU check (Colab/Jupyter)
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow version:", tf.__version__)
print("GPUs found:", len(gpus))

if gpus:
    print("GPU active: YES")
    for i, g in enumerate(gpus):
        print(f"  [{i}] {g}")
    print("Current device:", tf.test.gpu_device_name())
else:
    print("GPU active: NO")

In [4]:
# GPU type in Colab/Jupyter
import subprocess

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    )
    print("GPU info:")
    print(out.strip())
except Exception as e:
    print("Could not read GPU via nvidia-smi:", e)

In [5]:
# Install project requirements in Colab VM
import subprocess, sys
from pathlib import Path

REPO_DIR = Path(globals().get("EVAL_REPO_DIR", "/content/tcn_tape_vectorized_version_clean"))
REQ_FILE = REPO_DIR / "requirements.txt"

if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

print("Using python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", str(REQ_FILE)], check=True)

print("[OK] Requirements installed. If key libs were upgraded, restart runtime before continuing.")

## 2) Mount Drive and restore saved results zip

In [6]:
from pathlib import Path
EVAL_RESTORE_FROM_ZIP = True
EVAL_RESTORE_DIR = Path("/content/eval_restore")

if EVAL_RESTORE_FROM_ZIP:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    zip_path = Path(f"/content/drive/MyDrive/tcn_tape_vectorized_{RUN_ID}.zip")
    print("zip exists:", zip_path.exists())
    EVAL_RESTORE_DIR.mkdir(parents=True, exist_ok=True)
    !unzip -q -o "{zip_path}" -d "{EVAL_RESTORE_DIR}"
    print("[OK] Restored results from zip into", EVAL_RESTORE_DIR)
else:
    print("EVAL_RESTORE_FROM_ZIP=False")


In [7]:
from pathlib import Path
candidates = [
    EVAL_RESTORE_DIR / "tcn_fusion_results",
    Path(EVAL_REPO_DIR) / "tcn_fusion_results",
]
EVAL_RESULTS_ROOT = next((p for p in candidates if p.exists()), candidates[0])
print("[OK] EVAL_RESULTS_ROOT =", EVAL_RESULTS_ROOT)
print("   actor ckpts:", len(list(EVAL_RESULTS_ROOT.rglob("*_actor.weights.h5"))))

## 3) TF precision + imports

In [8]:
import tensorflow as tf
tf.keras.mixed_precision.set_global_policy("float32")
print(tf.keras.mixed_precision.global_policy())

In [9]:
import copy, json, re, shutil, time
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import get_active_config, apply_run5_overrides
from src.data_utils import DataProcessor
from src.notebook_helpers.tcn_phase1 import (
    prepare_phase1_dataset, create_experiment6_result_stub,
    evaluate_experiment6_checkpoint, load_training_metadata_into_config,
    build_evaluation_track_summary, Phase1Dataset,
    split_dataset_by_date, identify_covariance_columns,
)
print("[OK] All imports loaded")

## 4) Evaluation run settings

In [10]:
EVAL_RANDOM_SEED = 42
EVAL_FORCE_TEST_START_DATE = "2020-01-01"

# Deterministic evaluation windows
EVAL_START_DATES = ["2020-01-02", "2020-04-02", "2020-07-02", "2020-12-31"]
EVAL_HORIZONS_DAYS = [252, 504, 756, 1008]  # 1yr, 2yr, 3yr, 4yr

# Multi-horizon stochastic robustness
# test_log-style seeds are preserved; offsets can be fixed or randomized.
EVAL_NUM_STOCHASTIC_RUNS = 15
EVAL_STOCHASTIC_HORIZONS = {
    "1yr":  252,
    "2yr":  504,
    "3yr":  756,
    "4yr":  1008,
    "full": 9999,
}

# test_log parity seed family:
# evaluate_experiment6_checkpoint uses run_seed = random_seed + 100 + run_idx
EVAL_USE_TESTLOG_OFFSETS_AND_SEEDS = True
EVAL_STOCH_SEED_BASE_OFFSET = 500_000

# Offset controls
EVAL_STOCH_START_OFFSETS = [0, 63]  # anchor offsets
EVAL_RANDOMIZE_STOCH_OFFSETS = True
EVAL_STOCH_RANDOM_OFFSETS_PER_HORIZON = 2
EVAL_STOCH_RANDOM_INCLUDE_ANCHORS = True
EVAL_APPLY_OFFSETS_TO_FULL_HORIZON = False

# Optional fallback (legacy fixed-horizon base seeds)
EVAL_USE_FIXED_STOCH_SEEDS = False
EVAL_FIXED_STOCH_SEED_BASE_BY_HORIZON = {
    "1yr":  752042,
    "2yr":  1004042,
    "3yr":  1256042,
    "4yr":  1508042,
    "full": 1508042,
}

# Policy modes
EVAL_DETERMINISTIC_MODE = "mean"
EVAL_STOCHASTIC_MODE = "sample"

# Optional: pin exact metadata file for strict checkpoint/run parity.
EVAL_METADATA_PATH_OVERRIDE = None

# Asset universe (must match training)
EVAL_ASSET_UNIVERSE = ["MSFT", "GOOGL", "JPM", "JNJ", "XOM",
                       "PG", "NEE", "LIN", "CAT", "UNH"]

# Ensure Drive is mounted for output persistence in Colab.
EVAL_ENSURE_DRIVE_MOUNT_FOR_OUTPUT = True
if EVAL_ENSURE_DRIVE_MOUNT_FOR_OUTPUT and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"[WARN] Could not mount Drive for outputs: {e}")

# Output
EVAL_SAVE_LOGS = True
EVAL_SAVE_ARTIFACTS = True

DRIVE_OUTPUT_DIR = Path(f"/content/drive/MyDrive/robustness_ep398_{RUN_ID}")
LOCAL_OUTPUT_DIR = Path("/content/robustness_results")
OUTPUT_DIR = DRIVE_OUTPUT_DIR if Path("/content/drive/MyDrive").exists() else LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_hz_labels = list(EVAL_STOCHASTIC_HORIZONS.keys())
_offsets_per_h = {
    h: (EVAL_STOCH_START_OFFSETS if (h != "full" or EVAL_APPLY_OFFSETS_TO_FULL_HORIZON) else [0])
    for h in _hz_labels
}
EVAL_STOCH_BLOCKS_EXPECTED = sum(len(v) for v in _offsets_per_h.values())
EVAL_STOCH_TOTAL_RUNS_EXPECTED = EVAL_STOCH_BLOCKS_EXPECTED * EVAL_NUM_STOCHASTIC_RUNS

print("[OK] Settings configured")
print(f"   Checkpoint: {CHECKPOINT_TAG}")
print(f"   Det start dates: {EVAL_START_DATES}")
print(f"   Det horizons: {EVAL_HORIZONS_DAYS}")
print(f"   Stochastic horizons: {EVAL_STOCHASTIC_HORIZONS}")
print(f"   Offset mode: {'randomized' if EVAL_RANDOMIZE_STOCH_OFFSETS else 'fixed'} | anchors={EVAL_STOCH_START_OFFSETS}")
print(f"   Stochastic expected (min): {EVAL_STOCH_BLOCKS_EXPECTED} blocks x {EVAL_NUM_STOCHASTIC_RUNS} runs = {EVAL_STOCH_TOTAL_RUNS_EXPECTED} runs")
print(f"   Output dir: {OUTPUT_DIR}")
print(f"   Drive output dir: {DRIVE_OUTPUT_DIR}")
print(f"   Active output dir: {OUTPUT_DIR}")
print(f"   Output persistence: {'Drive' if OUTPUT_DIR == DRIVE_OUTPUT_DIR else 'Local'}")



## 5) Locate normalized CSV

In [11]:
target_dir = Path(EVAL_REPO_DIR) / "data"
target_dir.mkdir(parents=True, exist_ok=True)
target_path = target_dir / "master_features_NORMALIZED.csv"

candidates = []
candidates += list(EVAL_RESTORE_DIR.rglob("master_features_NORMALIZED.csv"))
candidates += list(EVAL_RESTORE_DIR.rglob("*normalized*.csv"))
seen, ordered = set(), []
for p in candidates:
    if p.suffix.lower() == ".csv" and str(p.resolve()) not in seen:
        seen.add(str(p.resolve())); ordered.append(p)
if not ordered:
    raise FileNotFoundError(f"No normalized CSV found under {EVAL_RESTORE_DIR}")
best = max(ordered, key=lambda p: p.stat().st_mtime)
shutil.copy2(best, target_path)
print("[OK] Normalized CSV:", best, "->", target_path)

## 6) Build eval config from training metadata + feature lock

In [12]:
def eval_extract_trained_state_layout(metadata_dict):
    arch = metadata_dict.get("Architecture_Settings", {}) or {}
    effective = arch.get("agent_params_effective", {}) or {}
    template = arch.get("agent_params_template", {}) or {}
    layout = effective.get("state_layout") or template.get("state_layout")
    if not isinstance(layout, dict): raise ValueError("state_layout not found")
    active_cols = layout.get("active_feature_columns")
    if not isinstance(active_cols, list) or not active_cols:
        raise ValueError("active_feature_columns missing")
    return layout, list(dict.fromkeys(active_cols))

def eval_apply_metadata_feature_lock(cfg, trained_cols):
    probe_cfg = copy.deepcopy(cfg)
    probe_cfg.setdefault("feature_params", {}).setdefault("feature_selection", {})
    probe_cfg["feature_params"]["feature_selection"]["disable_features"] = False
    probe_cfg["feature_params"]["feature_selection"]["disabled_features"] = []
    core = list(dict.fromkeys(DataProcessor(probe_cfg).get_feature_columns("phase1")))
    for c in trained_cols:
        if c not in core: core.append(c)
    gap = {c for c in core if c not in set(trained_cols)}
    existing = set(cfg.get("feature_params",{}).get("feature_selection",{}).get("disabled_features",[]))
    disabled = sorted(existing.union(gap))
    fs = cfg.setdefault("feature_params",{}).setdefault("feature_selection",{})
    fs["disable_features"] = True; fs["disabled_features"] = disabled
    return core, disabled

def eval_bind_trained_feature_layout(processor, trained_cols):
    cols = list(dict.fromkeys(trained_cols))
    base = processor.get_feature_columns
    def _locked(phase='phase1'):
        return list(cols) if str(phase).lower()=='phase1' else base(phase)
    processor.get_feature_columns = _locked
    return processor

# Load metadata
eval_config = copy.deepcopy(get_active_config("phase1"))
eval_logs_dir = EVAL_RESULTS_ROOT / "logs"
meta_files = sorted(eval_logs_dir.glob("*_metadata.json"), key=lambda p: p.stat().st_mtime, reverse=True)

meta_override = globals().get("EVAL_METADATA_PATH_OVERRIDE", None)
if meta_override:
    EVAL_METADATA_PATH = Path(meta_override)
    if not EVAL_METADATA_PATH.exists():
        raise FileNotFoundError(f"EVAL_METADATA_PATH_OVERRIDE not found: {EVAL_METADATA_PATH}")
else:
    if not meta_files:
        raise FileNotFoundError(f"No metadata files under: {eval_logs_dir}")
    EVAL_METADATA_PATH = meta_files[0]

print("[DOC] Metadata:", EVAL_METADATA_PATH)
with open(EVAL_METADATA_PATH, "r") as f:
    eval_metadata = json.load(f)

eval_config = load_training_metadata_into_config(EVAL_METADATA_PATH, copy.deepcopy(eval_config), verbose=True)

# Force test start
if EVAL_FORCE_TEST_START_DATE:
    ts = pd.to_datetime(EVAL_FORCE_TEST_START_DATE)
    eval_config["TRAIN_TEST_SPLIT_DATE"] = (ts - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# Hard requirements
eval_config.setdefault("feature_params",{}).setdefault("fundamental_features",{})["enabled"] = False
eval_config["feature_params"].setdefault("actuarial_params",{})["enabled"] = True
eval_config.setdefault("training_params",{})["num_parallel_envs"] = 1
eval_config["ASSET_TICKERS"] = list(EVAL_ASSET_UNIVERSE)
eval_config["NUM_ASSETS"] = len(EVAL_ASSET_UNIVERSE)
eval_config["agent_params"]["actor_critic_type"] = "TCN_FUSION"
eval_config["agent_params"]["use_fusion"] = True
eval_config["agent_params"]["use_attention"] = False

# Architecture defaults from metadata
ae = (eval_metadata.get("Architecture_Settings",{}) or {}).get("agent_params_effective",{}) or {}
for k,fb in {"fusion_cross_asset_mixer_enabled":True,"fusion_cross_asset_mixer_layers":1,
    "fusion_cross_asset_mixer_expansion":2.0,"fusion_cross_asset_mixer_dropout":0.1,
    "fusion_asset_identity_enabled":True,"fusion_context_cross_attention_enabled":False,
    "fusion_context_cross_attention_heads":4,"fusion_context_cross_attention_dropout":0.1,
    "fusion_per_asset_alpha_head":True,"fusion_alpha_head_hidden_dims":[128,64],
    "fusion_alpha_head_dropout":0.05,"recurrent_memory_enabled":False,
    "regime_conditioning_enabled":False,"state_augmentation_enabled":False,
    "distributional_critic_enabled":False,"distributional_num_quantiles":17,
    "dual_head_enabled":False}.items():
    eval_config["agent_params"][k] = ae.get(k, eval_config["agent_params"].get(k, fb))

ppo = eval_config["agent_params"].setdefault("ppo_params",{})
for k,fb in {"popart_enabled":False,"multi_horizon_reward_enabled":False,"dual_head_consistency_coef":0.0}.items():
    ppo.setdefault(k, fb)

# Feature lock
trained_state_layout, trained_active_feature_columns = eval_extract_trained_state_layout(eval_metadata)
eval_config["agent_params"]["state_layout"] = copy.deepcopy(trained_state_layout)
eval_config["agent_params"]["asset_feature_dim"] = int(trained_state_layout.get("asset_feature_dim",0) or 0)
eval_config["agent_params"]["global_feature_dim"] = int(trained_state_layout.get("global_feature_dim",0) or 0)
eval_config["agent_params"]["num_assets"] = int(trained_state_layout.get("num_assets",10) or 10)
core_all_cols, eval_disabled = eval_apply_metadata_feature_lock(eval_config, trained_active_feature_columns)
print(f"[OK] Config built | features: {len(trained_active_feature_columns)} | disabled: {len(eval_disabled)}")


# before running evaluate loop
# Keep evaluation deterministic and reproducible by default.

EVAL_FORCE_SINGLE_ENV = True
# If False, keep the values loaded from metadata/training config.
EVAL_APPLY_EXECUTION_TURNOVER_OVERRIDES = True

# De-constrained-aligned eval defaults
EVAL_OVERRIDE_ACTION_EXEC_BETA = 0.40
EVAL_OVERRIDE_TURNOVER_PENALTY = 0.05

# --- Run 5 overrides (comment out to revert to Run 4) ---
apply_run5_overrides(eval_config)

if EVAL_FORCE_SINGLE_ENV:
    eval_config.setdefault("training_params", {})["num_parallel_envs"] = 1

if EVAL_APPLY_EXECUTION_TURNOVER_OVERRIDES:
    eval_config.setdefault("training_params", {})["evaluation_action_execution_beta"] = EVAL_OVERRIDE_ACTION_EXEC_BETA
    eval_config.setdefault("training_params", {})["evaluation_turnover_penalty_scalar"] = EVAL_OVERRIDE_TURNOVER_PENALTY
    # Keep env mirror in sync for code paths that read environment params directly.
    eval_config.setdefault("environment_params", {})["action_execution_beta"] = EVAL_OVERRIDE_ACTION_EXEC_BETA

# If a global forced start date was requested, re-assert split after run5 override application.
if EVAL_FORCE_TEST_START_DATE:
    ts = pd.to_datetime(EVAL_FORCE_TEST_START_DATE)
    eval_config["TRAIN_TEST_SPLIT_DATE"] = (ts - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

print("eval num_parallel_envs:", eval_config.get("training_params", {}).get("num_parallel_envs"))
print("eval action_execution_beta:", eval_config.get("training_params", {}).get("evaluation_action_execution_beta"))
print("eval turnover_penalty_scalar:", eval_config.get("training_params", {}).get("evaluation_turnover_penalty_scalar"))
print("eval env.action_execution_beta:", eval_config.get("environment_params", {}).get("action_execution_beta"))
print("eval TRAIN_TEST_SPLIT_DATE:", eval_config.get("TRAIN_TEST_SPLIT_DATE"))

print("eval memory/regime/distributional:", {
    "recurrent_memory_enabled": eval_config.get("agent_params", {}).get("recurrent_memory_enabled"),
    "regime_conditioning_enabled": eval_config.get("agent_params", {}).get("regime_conditioning_enabled"),
    "state_augmentation_enabled": eval_config.get("agent_params", {}).get("state_augmentation_enabled"),
    "distributional_critic_enabled": eval_config.get("agent_params", {}).get("distributional_critic_enabled"),
    "distributional_num_quantiles": eval_config.get("agent_params", {}).get("distributional_num_quantiles"),
})
print("eval dual-head:", {
    "dual_head_enabled": eval_config.get("agent_params", {}).get("dual_head_enabled"),
    "dual_head_blend_schedule": eval_config.get("agent_params", {}).get("dual_head_blend_schedule"),
    "dual_head_eval_deterministic_rho": eval_config.get("agent_params", {}).get("dual_head_eval_deterministic_rho"),
    "dual_head_eval_stochastic_rho": eval_config.get("agent_params", {}).get("dual_head_eval_stochastic_rho"),
    "dual_head_projection_use_constraints": eval_config.get("agent_params", {}).get("dual_head_projection_use_constraints"),
})
print("eval popart/reward-decomp:", {
    "popart_enabled": eval_config.get("agent_params", {}).get("ppo_params", {}).get("popart_enabled"),
    "multi_horizon_reward_enabled": eval_config.get("agent_params", {}).get("ppo_params", {}).get("multi_horizon_reward_enabled"),
    "multi_horizon_reward_coef": eval_config.get("agent_params", {}).get("ppo_params", {}).get("multi_horizon_reward_coef"),
})


## 7) Build evaluation dataset

In [13]:
if "eval_phase1_data" in globals(): del eval_phase1_data

normalized_path = Path(EVAL_REPO_DIR) / "data" / "master_features_NORMALIZED.csv"
master_df_norm = pd.read_csv(normalized_path)
master_df_norm["Date"] = pd.to_datetime(master_df_norm["Date"], utc=True, errors="coerce").dt.tz_localize(None)
master_df_norm = master_df_norm.dropna(subset=["Date"]).sort_values(["Date","Ticker"]).reset_index(drop=True)
analysis_start = pd.to_datetime(eval_config.get("ANALYSIS_START_DATE","2003-09-02"))
analysis_end = pd.to_datetime(eval_config.get("ANALYSIS_END_DATE","2025-09-01"))
master_df_norm = master_df_norm[(master_df_norm["Date"]>=analysis_start)&(master_df_norm["Date"]<=analysis_end)].copy()
master_df_norm = master_df_norm[master_df_norm["Ticker"].isin(EVAL_ASSET_UNIVERSE)].copy()

split_date = eval_config.get("TRAIN_TEST_SPLIT_DATE")
train_df, test_df, train_end_date, test_start_date = split_dataset_by_date(master_df_norm, date_column="Date", split_date=split_date)
eval_processor = DataProcessor(eval_config)
eval_processor = eval_bind_trained_feature_layout(eval_processor, trained_active_feature_columns)
eval_phase1_data = Phase1Dataset(
    master_df=master_df_norm, train_df=train_df, test_df=test_df, scalers={},
    train_end_date=train_end_date, test_start_date=test_start_date,
    covariance_columns=identify_covariance_columns(master_df_norm.columns),
    data_processor=eval_processor)
print(f"[OK] Dataset built | Test: {test_df.shape} | {test_df['Date'].min()} to {test_df['Date'].max()}")

---
## 8) Locate ep398 checkpoint

In [14]:
hw_dir = EVAL_RESULTS_ROOT / "high_watermark_checkpoints"
actor_path = hw_dir / f"{CHECKPOINT_TAG}_actor.weights.h5"
critic_path = hw_dir / f"{CHECKPOINT_TAG}_critic.weights.h5"
assert actor_path.exists(), f"Missing: {actor_path}"
assert critic_path.exists(), f"Missing: {critic_path}"
experiment6 = create_experiment6_result_stub(
    exp_idx=6,
    results_root=str(EVAL_RESULTS_ROOT),
    random_seed=EVAL_RANDOM_SEED,
    checkpoint_path=str(hw_dir / CHECKPOINT_TAG),
    base_agent_params=eval_config.get("agent_params", {}),
)
print(f"[OK] Checkpoint: {CHECKPOINT_TAG}")

---
## 9) Deterministic Sweep — 4 Start Dates × 4 Horizons

In [21]:
# ============================================================================
# MULTI-HORIZON STOCHASTIC ROBUSTNESS
# Uses: er.stochastic_results (DataFrame) + er.stochastic_weights/actions/alphas
# Supports fixed offsets and randomized offsets per horizon.
# Output mode: stochastic-only custom logs (suppress verbose evaluator prints).
# ============================================================================

import io
import contextlib

# Logging controls
STOCH_ONLY_OUTPUT = True           # suppress evaluator's built-in deterministic/stochastic logs
STOCH_PRINT_EACH_RUN = True        # print one concise line per sampled run
STOCH_PRINT_HORIZON_SUMMARY = True

all_stoch_results = []
all_stoch_daily_data = {}    # key = (horizon_label, start_offset, run_name)
stoch_by_horizon = {}


def _make_phase1_slice(base_phase1, start_offset: int, horizon_days: int):
    test_df = base_phase1.test_df.copy()
    test_df["Date"] = pd.to_datetime(test_df["Date"])
    unique_dates = pd.Series(test_df["Date"].dropna().unique()).sort_values().reset_index(drop=True)

    if int(start_offset) >= len(unique_dates):
        return None, None

    end_idx = min(len(unique_dates), int(start_offset) + int(horizon_days))
    win_dates = unique_dates.iloc[int(start_offset):end_idx]
    if len(win_dates) == 0:
        return None, None

    d0 = pd.to_datetime(win_dates.iloc[0])
    d1 = pd.to_datetime(win_dates.iloc[-1])

    sliced_df = test_df[(test_df["Date"] >= d0) & (test_df["Date"] <= d1)].copy()
    if sliced_df.empty:
        return None, None

    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced_df
    phase1_slice.test_start_date = d0

    meta = {
        "start_date": str(d0.date()),
        "end_date": str(d1.date()),
        "n_days": int(len(win_dates)),
    }
    return phase1_slice, meta


def _sample_offsets(base_phase1, horizon_days: int, horizon_label: str, horizon_idx: int):
    use_offsets = (horizon_label != "full" or EVAL_APPLY_OFFSETS_TO_FULL_HORIZON)
    if not use_offsets:
        return [0]

    base_offsets = sorted({int(x) for x in EVAL_STOCH_START_OFFSETS if int(x) >= 0})
    if not EVAL_RANDOMIZE_STOCH_OFFSETS:
        return base_offsets if base_offsets else [0]

    test_df = base_phase1.test_df.copy()
    test_df["Date"] = pd.to_datetime(test_df["Date"])
    unique_dates = pd.Series(test_df["Date"].dropna().unique()).sort_values().reset_index(drop=True)
    n_dates = len(unique_dates)
    if n_dates == 0:
        return [0]

    max_offset = max(0, n_dates - 1)
    requested = max(int(EVAL_STOCH_RANDOM_OFFSETS_PER_HORIZON), 1)

    anchors = []
    if EVAL_STOCH_RANDOM_INCLUDE_ANCHORS:
        anchors = [o for o in base_offsets if o <= max_offset]

    remaining_k = max(0, requested - len(anchors))
    if remaining_k == 0:
        return sorted(set(anchors))

    rng_seed = int(EVAL_RANDOM_SEED + horizon_idx * 10_000 + int(horizon_days))
    rng = np.random.default_rng(rng_seed)
    pool = [i for i in range(max_offset + 1) if i not in set(anchors)]
    if not pool:
        return sorted(set(anchors))

    take = min(remaining_k, len(pool))
    sampled = sorted(rng.choice(np.array(pool, dtype=int), size=take, replace=False).tolist())
    return sorted(set(anchors + sampled))


def _pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _mean_std_text(df: pd.DataFrame, candidates, *, mult=1.0, fmt='.4f', suffix=''):
    col = _pick_col(df, candidates)
    if col is None:
        return 'n/a ± n/a'
    s = pd.to_numeric(df[col], errors='coerce') * mult
    m = s.mean()
    st = s.std()
    if pd.isna(m):
        return 'n/a ± n/a'
    m_txt = format(float(m), fmt)
    st_txt = format(float(st), fmt) if not pd.isna(st) else 'n/a'
    return f"{m_txt}{suffix} ± {st_txt}{suffix}"


def _fmt(x, fmt='.2f', suffix=''):
    try:
        xv = float(x)
    except Exception:
        return 'n/a'
    if pd.isna(xv):
        return 'n/a'
    return f"{format(xv, fmt)}{suffix}"


def _eval_quiet(*, experiment6, phase1_data, config, random_seed, checkpoint_path_override,
                deterministic_eval_mode, stochastic_eval_mode, num_eval_runs,
                stochastic_episode_length_limit, save_eval_logs, save_eval_artifacts):
    if not STOCH_ONLY_OUTPUT:
        return evaluate_experiment6_checkpoint(
            experiment6=experiment6,
            phase1_data=phase1_data,
            config=config,
            random_seed=random_seed,
            checkpoint_path_override=checkpoint_path_override,
            deterministic_eval_mode=deterministic_eval_mode,
            stochastic_eval_mode=stochastic_eval_mode,
            num_eval_runs=num_eval_runs,
            stochastic_episode_length_limit=stochastic_episode_length_limit,
            save_eval_logs=save_eval_logs,
            save_eval_artifacts=save_eval_artifacts,
        )

    buf_out, buf_err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
        er = evaluate_experiment6_checkpoint(
            experiment6=experiment6,
            phase1_data=phase1_data,
            config=config,
            random_seed=random_seed,
            checkpoint_path_override=checkpoint_path_override,
            deterministic_eval_mode=deterministic_eval_mode,
            stochastic_eval_mode=stochastic_eval_mode,
            num_eval_runs=num_eval_runs,
            stochastic_episode_length_limit=stochastic_episode_length_limit,
            save_eval_logs=save_eval_logs,
            save_eval_artifacts=save_eval_artifacts,
        )
    return er


total_horizons = len(EVAL_STOCHASTIC_HORIZONS)

for h_idx, (horizon_label, horizon_days) in enumerate(EVAL_STOCHASTIC_HORIZONS.items(), start=1):
    print(f"\n{'='*80}")
    print(f"STOCHASTIC HORIZON {h_idx}/{total_horizons}: {horizon_label} ({horizon_days}d) — {EVAL_NUM_STOCHASTIC_RUNS} runs")
    print(f"{'='*80}")

    offsets = _sample_offsets(eval_phase1_data, int(horizon_days), horizon_label, h_idx)
    print(f"   offsets used: {offsets}")

    horizon_rows = []
    horizon_run_counter = 0

    for start_offset in offsets:
        if globals().get("EVAL_USE_TESTLOG_OFFSETS_AND_SEEDS", False):
            horizon_seed = int(EVAL_RANDOM_SEED + EVAL_STOCH_SEED_BASE_OFFSET + int(horizon_days) * 1000 + int(start_offset))
        elif globals().get("EVAL_USE_FIXED_STOCH_SEEDS", False):
            fixed_map = globals().get("EVAL_FIXED_STOCH_SEED_BASE_BY_HORIZON", {}) or {}
            horizon_seed = int(fixed_map.get(horizon_label, EVAL_RANDOM_SEED + h_idx * 1000 + int(start_offset)))
        else:
            horizon_seed = int(EVAL_RANDOM_SEED + h_idx * 1000 + int(start_offset))

        seed_first = int(horizon_seed + 100)
        seed_last = int(horizon_seed + 100 + max(EVAL_NUM_STOCHASTIC_RUNS - 1, 0))
        print(f"   offset={int(start_offset):03d} | stochastic seeds: {seed_first}..{seed_last}")

        phase1_slice, meta = _make_phase1_slice(eval_phase1_data, int(start_offset), int(horizon_days))
        if phase1_slice is None:
            print(f"   [WARN] skip offset={int(start_offset):03d}: no valid slice")
            continue

        # one stochastic run per sampled start; we aggregate externally
        er = _eval_quiet(
            experiment6=experiment6,
            phase1_data=phase1_slice,
            config=eval_config,
            random_seed=horizon_seed,
            checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
            deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
            stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
            num_eval_runs=1,
            stochastic_episode_length_limit=min(int(horizon_days), int(meta["n_days"])),
            save_eval_logs=EVAL_SAVE_LOGS,
            save_eval_artifacts=EVAL_SAVE_ARTIFACTS,
        )

        hdf = er.stochastic_results.copy() if isinstance(er.stochastic_results, pd.DataFrame) else pd.DataFrame()
        if hdf.empty:
            print(f"   [WARN] no stochastic rows for {horizon_label} @ offset={int(start_offset):03d}")
            continue

        if "max_dd_pct" not in hdf.columns:
            mdd_col = _pick_col(hdf, ["max_drawdown", "max_drawdown_abs"])
            hdf["max_dd_pct"] = pd.to_numeric(hdf[mdd_col], errors='coerce') * 100.0 if mdd_col else np.nan
        if "ann_return_pct" not in hdf.columns:
            r_col = _pick_col(hdf, ["annualized_return", "annualized_return_pct"])
            hdf["ann_return_pct"] = pd.to_numeric(hdf[r_col], errors='coerce') * (100.0 if r_col == "annualized_return" else 1.0) if r_col else np.nan
        if "total_return_pct" not in hdf.columns:
            tr_col = _pick_col(hdf, ["total_return", "total_return_pct"])
            hdf["total_return_pct"] = pd.to_numeric(hdf[tr_col], errors='coerce') * (100.0 if tr_col == "total_return" else 1.0) if tr_col else np.nan
        if "volatility_ann" not in hdf.columns:
            v_col = _pick_col(hdf, ["volatility", "volatility_ann", "volatility_annualized"])
            hdf["volatility_ann"] = pd.to_numeric(hdf[v_col], errors='coerce') * (100.0 if v_col in ["volatility", "volatility_annualized"] else 1.0) if v_col else np.nan

        hdf["horizon"] = horizon_label
        hdf["horizon_days_limit"] = int(horizon_days)
        hdf["start_offset"] = int(start_offset)
        hdf["window_start_date"] = meta["start_date"]
        hdf["window_end_date"] = meta["end_date"]

        keep_cols = [
            "horizon", "horizon_days_limit", "start_offset", "window_start_date", "window_end_date",
            "run", "seed", "start_date", "market_regime", "days_traded",
            "total_return_pct", "ann_return_pct", "sharpe_ratio", "sortino_ratio",
            "max_dd_pct", "volatility_ann", "turnover", "win_rate",
            "turnover_step_mean", "turnover_step_p95", "turnover_step_max",
            "turnover_exceed_rate", "executed_to_raw_turnover_ratio",
        ]
        hflat = hdf[[c for c in keep_cols if c in hdf.columns]].copy()
        horizon_rows.append(hflat)
        all_stoch_results.extend(hflat.to_dict(orient="records"))

        # concise per-run stochastic log line
        horizon_run_counter += 1
        if STOCH_PRINT_EACH_RUN:
            rr = hflat.iloc[0]
            print(
                f"   [STOCH] {horizon_label} run={horizon_run_counter:03d} "
                f"start={rr.get('start_date', meta['start_date'])} seed={int(rr.get('seed', np.nan)) if pd.notna(rr.get('seed', np.nan)) else 'n/a'} "
                f"ret={_fmt(rr.get('total_return_pct', np.nan), '.2f', '%')} "
                f"sharpe={_fmt(rr.get('sharpe_ratio', np.nan), '.4f')} "
                f"mdd={_fmt(rr.get('max_dd_pct', np.nan), '.2f', '%')} "
                f"vol={_fmt(rr.get('volatility_ann', np.nan), '.2f', '%')} "
                f"turn={_fmt(rr.get('turnover', np.nan), '.2f') if pd.notna(rr.get('turnover', np.nan)) else 'n/a'}"
            )

        for run_idx, w_arr in enumerate(er.stochastic_weights or [], start=1):
            run_name = f"run_{run_idx:02d}"

            w_arr = np.asarray(w_arr) if w_arr is not None else np.empty((0, 0))
            a_arr = np.asarray(er.stochastic_actions[run_idx - 1]) if run_idx - 1 < len(er.stochastic_actions or []) else np.empty((0, 0))
            al_arr = np.asarray(er.stochastic_alphas[run_idx - 1]) if run_idx - 1 < len(er.stochastic_alphas or []) else np.empty((0, 0))

            n = len(w_arr)
            df_daily = pd.DataFrame({"step": np.arange(n)})
            df_daily["horizon"] = horizon_label
            df_daily["start_offset"] = int(start_offset)

            if w_arr.ndim == 2 and w_arr.shape[1] > 0:
                n_assets = len(EVAL_ASSET_UNIVERSE)
                for j, t in enumerate(EVAL_ASSET_UNIVERSE):
                    if j < w_arr.shape[1]:
                        df_daily[f"w_{t}"] = w_arr[:, j]
                if w_arr.shape[1] > n_assets:
                    df_daily["w_cash"] = w_arr[:, -1]

            if a_arr.ndim == 2 and len(a_arr) == n:
                for j in range(a_arr.shape[1]):
                    df_daily[f"a_{j}"] = a_arr[:, j]

            if al_arr.ndim == 2 and len(al_arr) == n:
                for j in range(al_arr.shape[1]):
                    df_daily[f"alpha_{j}"] = al_arr[:, j]

            meta_row = hdf[hdf["run"] == run_idx]
            if not meta_row.empty:
                meta_row = meta_row.iloc[0]
                df_daily["run"] = run_idx
                df_daily["start_date"] = meta_row.get("start_date")
                df_daily["market_regime"] = meta_row.get("market_regime")

            all_stoch_daily_data[(horizon_label, int(start_offset), run_name)] = df_daily

    horizon_df = pd.concat(horizon_rows, ignore_index=True) if horizon_rows else pd.DataFrame()
    stoch_by_horizon[horizon_label] = horizon_df

    if STOCH_PRINT_HORIZON_SUMMARY:
        print()
        if horizon_df.empty:
            print(f"   {horizon_label} SUMMARY: no rows")
        else:
            print(f"   {horizon_label} SUMMARY ({len(horizon_df)} runs across offsets={offsets}):")
            print(f"      Sharpe  — Mean: {_mean_std_text(horizon_df, ['sharpe_ratio', 'sharpe'], fmt='.4f')}")
            print(f"      MDD     — Mean: {_mean_std_text(horizon_df, ['max_dd_pct', 'max_drawdown', 'max_drawdown_abs'], mult=(1.0 if 'max_dd_pct' in horizon_df.columns else 100.0), fmt='.2f', suffix='%')}")
            print(f"      Return  — Mean: {_mean_std_text(horizon_df, ['ann_return_pct', 'annualized_return', 'annualized_return_pct'], mult=(1.0 if 'ann_return_pct' in horizon_df.columns else 100.0), fmt='.2f', suffix='%')}")
            print(f"      Days    — Mean: {_mean_std_text(horizon_df, ['days_traded'], fmt='.0f')}")

all_stoch_results_df = pd.DataFrame(all_stoch_results)
print(f"\n{'='*80}")
print(f"MULTI-HORIZON STOCHASTIC COMPLETE: {len(all_stoch_results_df)} total runs")
print(f"{'='*80}")



In [22]:
det_csv_path = OUTPUT_DIR / "ep398_deterministic_summary.csv"
det_results_df.to_csv(det_csv_path, index=False)
daily_dir = OUTPUT_DIR / "daily_data"
daily_dir.mkdir(parents=True, exist_ok=True)
for (sd,hd),df in det_daily_data.items():
    df.to_csv(daily_dir / f"ep398_det_{sd}_{hd}d_daily.csv", index=False)
print(f"[OK] Saved {len(det_results_df)} det summary + {len(det_daily_data)} daily files")

---
## 10) Multi-Horizon Stochastic Robustness — 30 Runs × 5 Horizons

Runs 30 stochastic evaluations at each of **5 different horizons** (1yr, 2yr, 3yr, 4yr, full).  
This produces **150 total stochastic runs** to verify that robustness holds  
across short, medium, and long investment horizons.

In [15]:
# ============================================================================
# MULTI-HORIZON STOCHASTIC ROBUSTNESS
# Uses: er.stochastic_results (DataFrame) + er.stochastic_weights/actions/alphas
# Supports fixed offsets and randomized offsets per horizon.
# ============================================================================

all_stoch_results = []
all_stoch_daily_data = {}    # key = (horizon_label, start_offset, run_name)
stoch_by_horizon = {}


def _make_phase1_slice(base_phase1, start_offset: int, horizon_days: int):
    test_df = base_phase1.test_df.copy()
    test_df["Date"] = pd.to_datetime(test_df["Date"])
    unique_dates = pd.Series(test_df["Date"].dropna().unique()).sort_values().reset_index(drop=True)

    if int(start_offset) >= len(unique_dates):
        return None, None

    end_idx = min(len(unique_dates), int(start_offset) + int(horizon_days))
    win_dates = unique_dates.iloc[int(start_offset):end_idx]
    if len(win_dates) == 0:
        return None, None

    d0 = pd.to_datetime(win_dates.iloc[0])
    d1 = pd.to_datetime(win_dates.iloc[-1])

    sliced_df = test_df[(test_df["Date"] >= d0) & (test_df["Date"] <= d1)].copy()
    if sliced_df.empty:
        return None, None

    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced_df
    phase1_slice.test_start_date = d0

    meta = {
        "start_date": str(d0.date()),
        "end_date": str(d1.date()),
        "n_days": int(len(win_dates)),
    }
    return phase1_slice, meta


def _sample_offsets(base_phase1, horizon_days: int, horizon_label: str, horizon_idx: int):
    use_offsets = (horizon_label != "full" or EVAL_APPLY_OFFSETS_TO_FULL_HORIZON)
    if not use_offsets:
        return [0]

    base_offsets = sorted({int(x) for x in EVAL_STOCH_START_OFFSETS if int(x) >= 0})
    if not EVAL_RANDOMIZE_STOCH_OFFSETS:
        return base_offsets if base_offsets else [0]

    test_df = base_phase1.test_df.copy()
    test_df["Date"] = pd.to_datetime(test_df["Date"])
    unique_dates = pd.Series(test_df["Date"].dropna().unique()).sort_values().reset_index(drop=True)
    n_dates = len(unique_dates)
    if n_dates == 0:
        return [0]

    max_offset = max(0, n_dates - 1)
    requested = max(int(EVAL_STOCH_RANDOM_OFFSETS_PER_HORIZON), 1)

    anchors = []
    if EVAL_STOCH_RANDOM_INCLUDE_ANCHORS:
        anchors = [o for o in base_offsets if o <= max_offset]

    remaining_k = max(0, requested - len(anchors))
    if remaining_k == 0:
        return sorted(set(anchors))

    rng_seed = int(EVAL_RANDOM_SEED + horizon_idx * 10_000 + int(horizon_days))
    rng = np.random.default_rng(rng_seed)
    pool = [i for i in range(max_offset + 1) if i not in set(anchors)]
    if not pool:
        return sorted(set(anchors))

    take = min(remaining_k, len(pool))
    sampled = sorted(rng.choice(np.array(pool, dtype=int), size=take, replace=False).tolist())
    return sorted(set(anchors + sampled))


def _pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _mean_std_text(df: pd.DataFrame, candidates, *, mult=1.0, fmt='.4f', suffix=''):
    col = _pick_col(df, candidates)
    if col is None:
        return 'n/a ± n/a'
    s = pd.to_numeric(df[col], errors='coerce') * mult
    m = s.mean()
    st = s.std()
    if pd.isna(m):
        return 'n/a ± n/a'
    m_txt = format(float(m), fmt)
    st_txt = format(float(st), fmt) if not pd.isna(st) else 'n/a'
    return f"{m_txt}{suffix} ± {st_txt}{suffix}"


total_horizons = len(EVAL_STOCHASTIC_HORIZONS)

for h_idx, (horizon_label, horizon_days) in enumerate(EVAL_STOCHASTIC_HORIZONS.items(), start=1):
    print(f"\n{'='*80}")
    print(f"STOCHASTIC HORIZON {h_idx}/{total_horizons}: {horizon_label} ({horizon_days}d) — {EVAL_NUM_STOCHASTIC_RUNS} runs")
    print(f"{'='*80}")

    offsets = _sample_offsets(eval_phase1_data, int(horizon_days), horizon_label, h_idx)
    print(f"   offsets used: {offsets}")

    horizon_rows = []

    for start_offset in offsets:
        if globals().get("EVAL_USE_TESTLOG_OFFSETS_AND_SEEDS", False):
            horizon_seed = int(EVAL_RANDOM_SEED + EVAL_STOCH_SEED_BASE_OFFSET + int(horizon_days) * 1000 + int(start_offset))
        elif globals().get("EVAL_USE_FIXED_STOCH_SEEDS", False):
            fixed_map = globals().get("EVAL_FIXED_STOCH_SEED_BASE_BY_HORIZON", {}) or {}
            horizon_seed = int(fixed_map.get(horizon_label, EVAL_RANDOM_SEED + h_idx * 1000 + int(start_offset)))
        else:
            horizon_seed = int(EVAL_RANDOM_SEED + h_idx * 1000 + int(start_offset))

        seed_first = int(horizon_seed + 100)
        seed_last = int(horizon_seed + 100 + max(EVAL_NUM_STOCHASTIC_RUNS - 1, 0))
        print(f"   offset={int(start_offset):03d} | stochastic seeds: {seed_first}..{seed_last}")
        print(
            f"   eval controls: beta={eval_config.get('training_params', {}).get('evaluation_action_execution_beta')}, "
            f"turnover_pen={eval_config.get('training_params', {}).get('evaluation_turnover_penalty_scalar')}, "
            f"env_beta={eval_config.get('environment_params', {}).get('action_execution_beta')}"
        )

        phase1_slice, meta = _make_phase1_slice(eval_phase1_data, int(start_offset), int(horizon_days))
        if phase1_slice is None:
            print(f"   [WARN] skip offset={int(start_offset):03d}: no valid slice")
            continue

        er = evaluate_experiment6_checkpoint(
            experiment6=experiment6,
            phase1_data=phase1_slice,
            config=eval_config,
            random_seed=horizon_seed,
            checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
            deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
            stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
            num_eval_runs=EVAL_NUM_STOCHASTIC_RUNS,
            stochastic_episode_length_limit=min(int(horizon_days), int(meta["n_days"])),
            save_eval_logs=EVAL_SAVE_LOGS,
            save_eval_artifacts=EVAL_SAVE_ARTIFACTS,
        )

        hdf = er.stochastic_results.copy() if isinstance(er.stochastic_results, pd.DataFrame) else pd.DataFrame()
        if hdf.empty:
            print(f"   [WARN] no stochastic rows for {horizon_label} @ offset={int(start_offset):03d}")
            continue

        # Normalize expected output columns across evaluator variants
        if "max_dd_pct" not in hdf.columns:
            mdd_col = _pick_col(hdf, ["max_drawdown", "max_drawdown_abs"])
            hdf["max_dd_pct"] = pd.to_numeric(hdf[mdd_col], errors='coerce') * 100.0 if mdd_col else np.nan
        if "ann_return_pct" not in hdf.columns:
            r_col = _pick_col(hdf, ["annualized_return", "annualized_return_pct"]) 
            hdf["ann_return_pct"] = pd.to_numeric(hdf[r_col], errors='coerce') * (100.0 if r_col == "annualized_return" else 1.0) if r_col else np.nan

        hdf["horizon"] = horizon_label
        hdf["horizon_days_limit"] = int(horizon_days)
        hdf["start_offset"] = int(start_offset)
        hdf["window_start_date"] = meta["start_date"]
        hdf["window_end_date"] = meta["end_date"]

        # %-scaled convenience columns
        if "total_return_pct" not in hdf.columns:
            tr_col = _pick_col(hdf, ["total_return", "total_return_pct"])
            hdf["total_return_pct"] = pd.to_numeric(hdf[tr_col], errors='coerce') * (100.0 if tr_col == "total_return" else 1.0) if tr_col else np.nan
        if "volatility_ann" not in hdf.columns:
            v_col = _pick_col(hdf, ["volatility", "volatility_ann", "volatility_annualized"])
            hdf["volatility_ann"] = pd.to_numeric(hdf[v_col], errors='coerce') * (100.0 if v_col in ["volatility", "volatility_annualized"] else 1.0) if v_col else np.nan

        keep_cols = [
            "horizon", "horizon_days_limit", "start_offset", "window_start_date", "window_end_date",
            "run", "seed", "start_date", "market_regime", "days_traded",
            "total_return_pct", "ann_return_pct", "sharpe_ratio", "sortino_ratio",
            "max_dd_pct", "volatility_ann", "turnover", "win_rate",
            "turnover_step_mean", "turnover_step_p95", "turnover_step_max",
            "turnover_exceed_rate", "executed_to_raw_turnover_ratio",
        ]
        hflat = hdf[[c for c in keep_cols if c in hdf.columns]].copy()
        horizon_rows.append(hflat)
        all_stoch_results.extend(hflat.to_dict(orient="records"))

        for run_idx, w_arr in enumerate(er.stochastic_weights or [], start=1):
            run_name = f"run_{run_idx:02d}"

            w_arr = np.asarray(w_arr) if w_arr is not None else np.empty((0, 0))
            a_arr = np.asarray(er.stochastic_actions[run_idx - 1]) if run_idx - 1 < len(er.stochastic_actions or []) else np.empty((0, 0))
            al_arr = np.asarray(er.stochastic_alphas[run_idx - 1]) if run_idx - 1 < len(er.stochastic_alphas or []) else np.empty((0, 0))

            n = len(w_arr)
            df_daily = pd.DataFrame({"step": np.arange(n)})
            df_daily["horizon"] = horizon_label
            df_daily["start_offset"] = int(start_offset)

            if w_arr.ndim == 2 and w_arr.shape[1] > 0:
                n_assets = len(EVAL_ASSET_UNIVERSE)
                for j, t in enumerate(EVAL_ASSET_UNIVERSE):
                    if j < w_arr.shape[1]:
                        df_daily[f"w_{t}"] = w_arr[:, j]
                if w_arr.shape[1] > n_assets:
                    df_daily["w_cash"] = w_arr[:, -1]

            if a_arr.ndim == 2 and len(a_arr) == n:
                for j in range(a_arr.shape[1]):
                    df_daily[f"a_{j}"] = a_arr[:, j]

            if al_arr.ndim == 2 and len(al_arr) == n:
                for j in range(al_arr.shape[1]):
                    df_daily[f"alpha_{j}"] = al_arr[:, j]

            meta_row = hdf[hdf["run"] == run_idx]
            if not meta_row.empty:
                meta_row = meta_row.iloc[0]
                df_daily["run"] = run_idx
                df_daily["start_date"] = meta_row.get("start_date")
                df_daily["market_regime"] = meta_row.get("market_regime")

            all_stoch_daily_data[(horizon_label, int(start_offset), run_name)] = df_daily

    horizon_df = pd.concat(horizon_rows, ignore_index=True) if horizon_rows else pd.DataFrame()
    stoch_by_horizon[horizon_label] = horizon_df

    print()
    if horizon_df.empty:
        print(f"   {horizon_label} SUMMARY: no rows")
    else:
        print(f"   {horizon_label} SUMMARY ({len(horizon_df)} runs across offsets={offsets}):")
        print(f"      Sharpe  — Mean: {_mean_std_text(horizon_df, ['sharpe_ratio', 'sharpe'], fmt='.4f')}")
        print(f"      MDD     — Mean: {_mean_std_text(horizon_df, ['max_dd_pct', 'max_drawdown', 'max_drawdown_abs'], mult=(1.0 if 'max_dd_pct' in horizon_df.columns else 100.0), fmt='.2f', suffix='%')}")
        print(f"      Return  — Mean: {_mean_std_text(horizon_df, ['ann_return_pct', 'annualized_return', 'annualized_return_pct'], mult=(1.0 if 'ann_return_pct' in horizon_df.columns else 100.0), fmt='.2f', suffix='%')}")
        print(f"      Days    — Mean: {_mean_std_text(horizon_df, ['days_traded'], fmt='.0f')}")

all_stoch_results_df = pd.DataFrame(all_stoch_results)
print(f"\n{'='*80}")
print(f"MULTI-HORIZON STOCHASTIC COMPLETE: {len(all_stoch_results_df)} total runs")
print(f"{'='*80}")



In [16]:
# Save all stochastic results
stoch_csv_path = OUTPUT_DIR / "ep398_stochastic_all_horizons.csv"
all_stoch_results_df.to_csv(stoch_csv_path, index=False)
print(f"[OK] Saved combined stochastic: {stoch_csv_path} ({len(all_stoch_results_df)} rows)")

# Per-horizon summaries
for hl, hdf in stoch_by_horizon.items():
    hdf.to_csv(OUTPUT_DIR / f"ep398_stochastic_{hl}.csv", index=False)

# Daily data
stoch_daily_dir = OUTPUT_DIR / "stochastic_daily"
stoch_daily_dir.mkdir(parents=True, exist_ok=True)

saved_daily = 0
for key, df in all_stoch_daily_data.items():
    if not isinstance(key, tuple):
        continue
    if len(key) == 3:
        hl, off, tn = key
        fname = f"ep398_stoch_{hl}_off{int(off):03d}_{tn}_daily.csv"
    elif len(key) == 2:
        hl, tn = key
        fname = f"ep398_stoch_{hl}_{tn}_daily.csv"
    else:
        fname = f"ep398_stoch_{'_'.join(map(str, key))}_daily.csv"

    df.to_csv(stoch_daily_dir / fname, index=False)
    saved_daily += 1

print(f"[OK] Saved {saved_daily} daily files + {len(stoch_by_horizon)} horizon CSVs")



In [ ]:
# ============================================================================
# PURE RANDOM-START STOCHASTIC ROBUSTNESS (NO OFFSETS)
# Stratified random start-date sampling across full OOS window per horizon
# - Suppresses verbose evaluator logs
# - Prints concise stochastic run lines
# - Prints one aggregate summary per horizon after all runs
# - Reports BOTH total and annualized returns
# ============================================================================

import copy
import io
import contextlib
import numpy as np
import pandas as pd

# ---------- knobs ----------
RND_HORIZONS = {
    "1yr": 252,
    "2yr": 504,
    "3yr": 756,
    "4yr": 1008,
}
RND_RUNS_PER_HORIZON = 40
RND_STRATA_PER_HORIZON = 10
RND_BASE_SEED = 42
RND_SAVE_LOGS = False
RND_SAVE_ARTIFACTS = False

RND_SUPPRESS_EVAL_VERBOSE = True
RND_PRINT_EACH_RUN = True
RND_PRINT_HORIZON_SUMMARY = True

# ---------- helpers ----------
def _unique_test_dates(base_phase1):
    t = base_phase1.test_df.copy()
    t["Date"] = pd.to_datetime(t["Date"])
    return pd.Series(t["Date"].dropna().unique()).sort_values().reset_index(drop=True)

def _make_phase1_slice_from_start_idx(base_phase1, start_idx: int, horizon_days: int):
    t = base_phase1.test_df.copy()
    t["Date"] = pd.to_datetime(t["Date"])
    u = _unique_test_dates(base_phase1)

    if start_idx < 0 or start_idx >= len(u):
        return None, None
    end_idx = min(len(u), start_idx + horizon_days)
    win = u.iloc[start_idx:end_idx]
    if len(win) < horizon_days:
        return None, None

    d0 = pd.to_datetime(win.iloc[0])
    d1 = pd.to_datetime(win.iloc[-1])

    sliced = t[(t["Date"] >= d0) & (t["Date"] <= d1)].copy()
    if sliced.empty:
        return None, None

    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced
    phase1_slice.test_start_date = d0

    meta = {"start_date": str(d0.date()), "end_date": str(d1.date()), "n_days": int(len(win))}
    return phase1_slice, meta

def _stratified_start_indices(n_valid: int, n_samples: int, n_strata: int, seed: int):
    rng = np.random.default_rng(seed)
    n_samples = min(n_samples, n_valid)
    n_strata = max(1, min(n_strata, n_valid))

    edges = np.linspace(0, n_valid, n_strata + 1, dtype=int)
    base = n_samples // n_strata
    rem = n_samples % n_strata

    picks = []
    for s in range(n_strata):
        lo, hi = edges[s], edges[s + 1]
        pool = np.arange(lo, hi)
        if len(pool) == 0:
            continue
        k = min(len(pool), base + (1 if s < rem else 0))
        if k > 0:
            picks.extend(rng.choice(pool, size=k, replace=False).tolist())

    if len(picks) < n_samples:
        remaining = np.setdiff1d(np.arange(n_valid), np.array(picks, dtype=int), assume_unique=False)
        need = n_samples - len(picks)
        if len(remaining) > 0:
            picks.extend(rng.choice(remaining, size=min(need, len(remaining)), replace=False).tolist())

    return sorted(picks[:n_samples])

def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _num_series(df, candidates, mult=1.0):
    c = _pick_col(df, candidates)
    if c is None:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[c], errors="coerce") * mult

def _pctile(s, q):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return np.nan if len(s) == 0 else float(np.percentile(s, q))

def _cvar_left_tail(s, alpha=0.10):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    cutoff = np.percentile(s, alpha * 100.0)
    tail = s[s <= cutoff]
    return float(tail.mean()) if len(tail) else np.nan

def _fmt(x, d=2, suffix=""):
    try:
        xv = float(x)
        if pd.isna(xv):
            return "n/a"
        return f"{xv:.{d}f}{suffix}"
    except Exception:
        return "n/a"

def _run_eval_quiet(**kwargs):
    if not RND_SUPPRESS_EVAL_VERBOSE:
        return evaluate_experiment6_checkpoint(**kwargs)
    buf_out, buf_err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
        er = evaluate_experiment6_checkpoint(**kwargs)
    return er

def _extract_sto_row(sto: pd.DataFrame):
    row = sto.iloc[0].copy()

    # Total return (%)
    if "total_return_pct" not in row.index:
        if "total_return" in row.index:
            row["total_return_pct"] = pd.to_numeric(row["total_return"], errors="coerce") * 100.0
        elif "total_return_ratio" in row.index:
            row["total_return_pct"] = pd.to_numeric(row["total_return_ratio"], errors="coerce") * 100.0

    # Annualized return (%)
    if "ann_return_pct" not in row.index:
        if "annualized_return" in row.index:
            row["ann_return_pct"] = pd.to_numeric(row["annualized_return"], errors="coerce") * 100.0
        elif "annualized_return_pct" in row.index:
            row["ann_return_pct"] = pd.to_numeric(row["annualized_return_pct"], errors="coerce")

    # MDD (%)
    if "max_dd_pct" not in row.index:
        if "max_drawdown" in row.index:
            row["max_dd_pct"] = pd.to_numeric(row["max_drawdown"], errors="coerce") * 100.0
        elif "max_drawdown_abs" in row.index:
            row["max_dd_pct"] = pd.to_numeric(row["max_drawdown_abs"], errors="coerce") * 100.0

    # Volatility ann (%)
    if "volatility_ann" not in row.index:
        if "volatility" in row.index:
            row["volatility_ann"] = pd.to_numeric(row["volatility"], errors="coerce") * 100.0
        elif "volatility_annualized" in row.index:
            row["volatility_ann"] = pd.to_numeric(row["volatility_annualized"], errors="coerce") * 100.0

    return row

# ---------- run ----------
random_stoch_rows = []
random_stoch_daily = {}
u_all = _unique_test_dates(eval_phase1_data)

for h_i, (h_label, h_days) in enumerate(RND_HORIZONS.items(), start=1):
    max_start_idx = len(u_all) - h_days
    if max_start_idx < 0:
        print(f"[WARN] {h_label}: horizon too long for available OOS dates")
        continue

    valid_count = max_start_idx + 1
    start_idxs = _stratified_start_indices(
        n_valid=valid_count,
        n_samples=RND_RUNS_PER_HORIZON,
        n_strata=RND_STRATA_PER_HORIZON,
        seed=RND_BASE_SEED + h_i * 10000 + h_days,
    )

    print(f"\n{'='*80}")
    print(f"{h_label} ({h_days}d): {len(start_idxs)} random starts (stratified)")
    print(f"valid start pool: {valid_count} | first={u_all.iloc[0].date()} | last={u_all.iloc[max_start_idx].date()}")
    print(f"{'='*80}")

    horizon_rows = []

    for j, s_idx in enumerate(start_idxs, start=1):
        phase1_slice, meta = _make_phase1_slice_from_start_idx(eval_phase1_data, s_idx, h_days)
        if phase1_slice is None:
            continue

        seed = int(RND_BASE_SEED + h_days * 100000 + j)

        er = _run_eval_quiet(
            experiment6=experiment6,
            phase1_data=phase1_slice,
            config=eval_config,
            random_seed=seed,
            checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
            deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
            stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
            num_eval_runs=1,
            stochastic_episode_length_limit=h_days,
            save_eval_logs=RND_SAVE_LOGS,
            save_eval_artifacts=RND_SAVE_ARTIFACTS,
        )

        sto = er.stochastic_results if isinstance(er.stochastic_results, pd.DataFrame) else pd.DataFrame()
        if sto.empty:
            continue

        row = _extract_sto_row(sto).to_dict()
        row["horizon"] = h_label
        row["horizon_days"] = h_days
        row["sample_id"] = j
        row["sampled_start_idx"] = int(s_idx)
        row["forced_start_date"] = meta["start_date"]
        row["window_end_date"] = meta["end_date"]

        random_stoch_rows.append(row)
        horizon_rows.append(row)

        if RND_PRINT_EACH_RUN:
            print(
                f"[{h_label} {j:03d}/{len(start_idxs)}] "
                f"start={row.get('start_date', meta['start_date'])} "
                f"seed={int(row['seed']) if pd.notna(row.get('seed', np.nan)) else 'n/a'} | "
                f"tot={_fmt(row.get('total_return_pct', np.nan), 2, '%')} | "
                f"ann={_fmt(row.get('ann_return_pct', np.nan), 2, '%')} | "
                f"sh={_fmt(row.get('sharpe_ratio', row.get('sharpe', np.nan)), 4)} | "
                f"so={_fmt(row.get('sortino_ratio', row.get('sortino', np.nan)), 4)} | "
                f"mdd={_fmt(row.get('max_dd_pct', np.nan), 2, '%')} | "
                f"vol={_fmt(row.get('volatility_ann', np.nan), 2, '%')} | "
                f"turn={_fmt(row.get('turnover', np.nan), 4)}"
            )

        if er.stochastic_weights and len(er.stochastic_weights) >= 1:
            w = np.asarray(er.stochastic_weights[0])
            a = np.asarray(er.stochastic_actions[0]) if er.stochastic_actions and len(er.stochastic_actions) >= 1 else np.empty((0,0))
            al = np.asarray(er.stochastic_alphas[0]) if er.stochastic_alphas and len(er.stochastic_alphas) >= 1 else np.empty((0,0))

            n = len(w)
            dd = pd.DataFrame({"step": np.arange(n), "horizon": h_label, "sample_id": j, "start_date": meta["start_date"]})
            if w.ndim == 2 and w.shape[1] > 0:
                for k, tkr in enumerate(EVAL_ASSET_UNIVERSE):
                    if k < w.shape[1]:
                        dd[f"w_{tkr}"] = w[:, k]
                if w.shape[1] > len(EVAL_ASSET_UNIVERSE):
                    dd["w_cash"] = w[:, -1]
            if a.ndim == 2 and len(a) == n:
                for k in range(a.shape[1]):
                    dd[f"a_{k}"] = a[:, k]
            if al.ndim == 2 and len(al) == n:
                for k in range(al.shape[1]):
                    dd[f"alpha_{k}"] = al[:, k]

            random_stoch_daily[(h_label, j)] = dd

    # one summary per horizon (after all runs in that horizon)
    if RND_PRINT_HORIZON_SUMMARY:
        hdf = pd.DataFrame(horizon_rows)
        if hdf.empty:
            print(f"[{h_label}] no completed runs")
        else:
            s = _num_series(hdf, ["sharpe_ratio", "sharpe"])
            so = _num_series(hdf, ["sortino_ratio", "sortino"])
            d = _num_series(hdf, ["max_dd_pct", "max_drawdown", "max_drawdown_abs"],
                            mult=(1.0 if "max_dd_pct" in hdf.columns else 100.0))
            ttot = _num_series(hdf, ["total_return_pct", "total_return"],
                               mult=(1.0 if "total_return_pct" in hdf.columns else 100.0))
            tann = _num_series(hdf, ["ann_return_pct", "annualized_return", "annualized_return_pct"],
                               mult=(1.0 if "ann_return_pct" in hdf.columns else 100.0))

            calmar = tann / d.replace(0, np.nan)

            print(f"\n[{h_label}] SUMMARY ({len(hdf)} runs)")
            print(f"  TotalRet mean±std: {ttot.mean():.2f}% ± {ttot.std():.2f}%")
            print(f"  AnnRet   mean±std: {tann.mean():.2f}% ± {tann.std():.2f}%")
            print(f"  Sharpe   mean±std: {s.mean():.4f} ± {s.std():.4f}")
            print(f"  Sortino  mean±std: {so.mean():.4f} ± {so.std():.4f}")
            print(f"  MDD      mean±std: {d.mean():.2f}% ± {d.std():.2f}%")
            print(f"  Calmar   mean:     {calmar.mean():.4f}")
            print(f"  Tail (AnnRet p10 / CVaR10): {_pctile(tann, 10):.2f}% / {_cvar_left_tail(tann, 0.10):.2f}%")

# ---------- results ----------
random_stoch_df = pd.DataFrame(random_stoch_rows)
print(f"\n[OK] Completed stochastic random-start runs: {len(random_stoch_df)}")

if not random_stoch_df.empty:
    sharpe = _num_series(random_stoch_df, ["sharpe_ratio", "sharpe"])
    sortino = _num_series(random_stoch_df, ["sortino_ratio", "sortino"])
    mdd = _num_series(random_stoch_df, ["max_dd_pct", "max_drawdown", "max_drawdown_abs"],
                      mult=(1.0 if "max_dd_pct" in random_stoch_df.columns else 100.0))
    tret = _num_series(random_stoch_df, ["total_return_pct", "total_return"],
                       mult=(1.0 if "total_return_pct" in random_stoch_df.columns else 100.0))
    annr = _num_series(random_stoch_df, ["ann_return_pct", "annualized_return", "annualized_return_pct"],
                       mult=(1.0 if "ann_return_pct" in random_stoch_df.columns else 100.0))

    print(f"[ALL] TotalRet mean±std: {tret.mean():.2f}% ± {tret.std():.2f}%")
    print(f"[ALL] AnnRet   mean±std: {annr.mean():.2f}% ± {annr.std():.2f}%")
    print(f"[ALL] Sharpe   mean±std: {sharpe.mean():.4f} ± {sharpe.std():.4f}")
    print(f"[ALL] Sortino  mean±std: {sortino.mean():.4f} ± {sortino.std():.4f}")
    print(f"[ALL] MDD      mean±std: {mdd.mean():.2f}% ± {mdd.std():.2f}%")

    rows = []
    for h, g in random_stoch_df.groupby("horizon"):
        s = _num_series(g, ["sharpe_ratio", "sharpe"])
        so = _num_series(g, ["sortino_ratio", "sortino"])
        d = _num_series(g, ["max_dd_pct", "max_drawdown", "max_drawdown_abs"],
                        mult=(1.0 if "max_dd_pct" in g.columns else 100.0))
        tr = _num_series(g, ["total_return_pct", "total_return"],
                         mult=(1.0 if "total_return_pct" in g.columns else 100.0))
        ar = _num_series(g, ["ann_return_pct", "annualized_return", "annualized_return_pct"],
                         mult=(1.0 if "ann_return_pct" in g.columns else 100.0))
        cal = ar / d.replace(0, np.nan)

        rows.append({
            "horizon": h,
            "n_runs": len(g),

            "total_ret_mean_pct": tr.mean(),
            "total_ret_std_pct": tr.std(),
            "total_ret_p10_pct": _pctile(tr, 10),
            "total_ret_p50_pct": _pctile(tr, 50),
            "total_ret_p90_pct": _pctile(tr, 90),
            "total_ret_cvar10_pct": _cvar_left_tail(tr, 0.10),

            "ann_ret_mean_pct": ar.mean(),
            "ann_ret_std_pct": ar.std(),
            "ann_ret_p10_pct": _pctile(ar, 10),
            "ann_ret_p50_pct": _pctile(ar, 50),
            "ann_ret_p90_pct": _pctile(ar, 90),
            "ann_ret_cvar10_pct": _cvar_left_tail(ar, 0.10),

            "sharpe_mean": s.mean(),
            "sharpe_std": s.std(),
            "sharpe_p10": _pctile(s, 10),
            "sharpe_p50": _pctile(s, 50),
            "sharpe_p90": _pctile(s, 90),

            "sortino_mean": so.mean(),
            "mdd_mean_pct": d.mean(),
            "mdd_std_pct": d.std(),
            "mdd_p90_pct": _pctile(d, 90),

            "calmar_mean": cal.mean(),
            "start_date_min": pd.to_datetime(g["forced_start_date"]).min().date(),
            "start_date_max": pd.to_datetime(g["forced_start_date"]).max().date(),
        })

    random_stoch_summary_df = pd.DataFrame(rows).sort_values("horizon")
    display(random_stoch_summary_df)

    random_stoch_df.to_csv(OUTPUT_DIR / "ep398_random_start_stochastic_all.csv", index=False)
    random_stoch_summary_df.to_csv(OUTPUT_DIR / "ep398_random_start_stochastic_summary.csv", index=False)
    print("[OK] Saved random-start stochastic outputs to", OUTPUT_DIR)

---
## 11) Cross-Horizon Stochastic Analysis

Compare robustness metrics across investment horizons to show  
whether the policy degrades, remains stable, or improves over longer periods.

In [17]:
# ============================================================================
# CROSS-HORIZON COMPARISON TABLE
# ============================================================================

def _pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _series(df: pd.DataFrame, candidates, mult=1.0):
    c = _pick_col(df, candidates)
    if c is None:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[c], errors="coerce") * mult


horizon_summary = []
for hl, hdf in stoch_by_horizon.items():
    if hdf.empty:
        continue

    sharpe = _series(hdf, ["sharpe_ratio", "sharpe"])
    mdd = _series(hdf, ["max_dd_pct", "max_drawdown", "max_drawdown_abs"], mult=(1.0 if "max_dd_pct" in hdf.columns else 100.0))
    ann_ret = _series(hdf, ["ann_return_pct", "annualized_return", "annualized_return_pct"], mult=(1.0 if "ann_return_pct" in hdf.columns else 100.0))
    turnover = _series(hdf, ["turnover"])  # already in evaluator-native units (usually %)
    win_rate = _series(hdf, ["win_rate"])  # evaluator-native units
    days = _series(hdf, ["days_traded"])

    offsets_used = sorted(int(x) for x in pd.to_numeric(hdf.get("start_offset", pd.Series(dtype=float)), errors="coerce").dropna().unique()) if "start_offset" in hdf.columns else []

    horizon_summary.append({
        "Horizon": hl,
        "N Runs": int(len(hdf)),
        "Offsets Used": ",".join(map(str, offsets_used)) if offsets_used else "n/a",
        "N Offsets": int(len(offsets_used)) if offsets_used else 0,
        "Sharpe Mean": f"{sharpe.mean():.3f}" if not sharpe.empty else "n/a",
        "Sharpe Std": f"{sharpe.std():.3f}" if not sharpe.empty else "n/a",
        "Sharpe Min": f"{sharpe.min():.3f}" if not sharpe.empty else "n/a",
        "Sharpe Max": f"{sharpe.max():.3f}" if not sharpe.empty else "n/a",
        "MDD Mean %": f"{mdd.mean():.1f}" if not mdd.empty else "n/a",
        "MDD Std %": f"{mdd.std():.1f}" if not mdd.empty else "n/a",
        "Ann Ret Mean %": f"{ann_ret.mean():.1f}" if not ann_ret.empty else "n/a",
        "Ann Ret Std %": f"{ann_ret.std():.1f}" if not ann_ret.empty else "n/a",
        "Turnover Mean": f"{turnover.mean():.4f}" if not turnover.empty else "n/a",
        "Win Rate Mean": f"{win_rate.mean():.2f}" if not win_rate.empty else "n/a",
        "Days Mean": f"{days.mean():.0f}" if not days.empty else "n/a",
        "Sharpe>0 %": f"{(sharpe > 0).mean()*100:.0f}" if not sharpe.empty else "n/a",
        "Sharpe>0.5 %": f"{(sharpe > 0.5).mean()*100:.0f}" if not sharpe.empty else "n/a",
        "Sharpe>1.0 %": f"{(sharpe > 1.0).mean()*100:.0f}" if not sharpe.empty else "n/a",
    })

cross_horizon_df = pd.DataFrame(horizon_summary)
print("\n" + "=" * 80)
print("CROSS-HORIZON STOCHASTIC ROBUSTNESS SUMMARY")
print("=" * 80)
display(cross_horizon_df)
cross_horizon_df.to_csv(OUTPUT_DIR / "ep398_cross_horizon_summary.csv", index=False)



In [18]:
# ============================================================================
# CROSS-HORIZON VISUALIZATIONS
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

horizon_order = list(EVAL_STOCHASTIC_HORIZONS.keys())
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    f"Cross-Horizon Stochastic Robustness — ep398 ({EVAL_NUM_STOCHASTIC_RUNS} runs/block, offsets={EVAL_STOCH_START_OFFSETS})",
    fontsize=14,
    fontweight="bold",
)

# Panel A: Sharpe box plot
ax = axes[0, 0]
sharpe_data = []
for h in horizon_order:
    hdf = stoch_by_horizon.get(h, pd.DataFrame())
    sharpe_col = "sharpe_ratio" if "sharpe_ratio" in hdf.columns else "sharpe"
    sharpe_data.append(hdf[sharpe_col].dropna().values if (not hdf.empty and sharpe_col in hdf.columns) else np.array([]))

bp = ax.boxplot(sharpe_data, labels=horizon_order, patch_artist=True, widths=0.6)
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(y=0, color="red", linestyle="--", alpha=0.5, label="Sharpe = 0")
ax.axhline(y=1, color="green", linestyle="--", alpha=0.5, label="Sharpe = 1")
ax.set_ylabel("Sharpe Ratio")
ax.set_title("A) Sharpe Ratio by Horizon")
ax.legend(fontsize=8)

# Panel B: Max Drawdown box plot
ax = axes[0, 1]
mdd_data = [stoch_by_horizon[h]["max_dd_pct"].dropna().values if h in stoch_by_horizon and not stoch_by_horizon[h].empty else np.array([]) for h in horizon_order]
bp = ax.boxplot(mdd_data, labels=horizon_order, patch_artist=True, widths=0.6)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Max Drawdown (%)")
ax.set_title("B) Maximum Drawdown by Horizon")

# Panel C: Annualized Return box plot
ax = axes[1, 0]
ret_data = [stoch_by_horizon[h]["ann_return_pct"].dropna().values if h in stoch_by_horizon and not stoch_by_horizon[h].empty else np.array([]) for h in horizon_order]
bp = ax.boxplot(ret_data, labels=horizon_order, patch_artist=True, widths=0.6)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(y=0, color="red", linestyle="--", alpha=0.5)
ax.set_ylabel("Annualized Return (%)")
ax.set_title("C) Annualized Return by Horizon")

# Panel D: Sharpe > threshold hit rates
ax = axes[1, 1]
thresholds = [0, 0.5, 1.0]
x = np.arange(len(horizon_order))
width = 0.25
for i, thr in enumerate(thresholds):
    rates = []
    for h in horizon_order:
        hdf = stoch_by_horizon.get(h, pd.DataFrame())
        sharpe_col = "sharpe_ratio" if "sharpe_ratio" in hdf.columns else "sharpe"
        rate = (hdf[sharpe_col] > thr).mean() * 100 if (not hdf.empty and sharpe_col in hdf.columns) else 0.0
        rates.append(rate)
    ax.bar(x + i * width, rates, width, label=f"Sharpe > {thr}", alpha=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels(horizon_order)
ax.set_ylabel("Hit Rate (%)")
ax.set_title("D) Sharpe Threshold Hit Rates")
ax.legend(fontsize=8)
ax.set_ylim(0, 105)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "ep398_cross_horizon_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Saved cross-horizon figure")



In [19]:
# ============================================================================
# STOCHASTIC FAN VIEW (per horizon)
# Uses per-step portfolio_value if available; otherwise shows placeholder note.
# ============================================================================

fig, axes = plt.subplots(1, len(horizon_order), figsize=(5 * len(horizon_order), 5), sharey=True)
if len(horizon_order) == 1:
    axes = [axes]

fig.suptitle(
    f"Stochastic Equity Curves — {EVAL_NUM_STOCHASTIC_RUNS} Runs/Block, Offsets={EVAL_STOCH_START_OFFSETS}",
    fontsize=14,
    fontweight="bold",
)

total_curves = 0
for ax, hl in zip(axes, horizon_order):
    curves_here = 0

    for key, df in all_stoch_daily_data.items():
        if not isinstance(key, tuple):
            continue
        h = key[0]
        if h != hl:
            continue
        if df is None or len(df) == 0:
            continue

        pv_col = None
        for c in ["portfolio_value", "equity", "equity_curve", "pv"]:
            if c in df.columns:
                pv_col = c
                break
        if pv_col is None:
            continue

        pv = pd.to_numeric(df[pv_col], errors="coerce").dropna()
        if len(pv) < 2:
            continue

        norm = pv.values / float(pv.iloc[0])
        ax.plot(
            np.arange(len(norm)),
            norm,
            alpha=0.25,
            linewidth=0.8,
            color=colors[horizon_order.index(hl) % len(colors)],
        )
        curves_here += 1

    ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(f"{hl}", fontsize=11)
    ax.set_xlabel("Trading Days")
    if ax == axes[0]:
        ax.set_ylabel("Normalized Portfolio Value")

    if curves_here == 0:
        ax.text(
            0.5,
            0.5,
            "No per-step\nportfolio_value\nin stochastic artifacts",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=9,
            alpha=0.8,
        )
    total_curves += curves_here

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "ep398_stochastic_fan_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Saved fan chart | curves plotted: {total_curves}")



---
## 12) Full OOS Single Pass — Primary Equity Curve

In [20]:
print("Running full-horizon OOS evaluation (2020-01-02 -> end of data)...")
full_oos_eval = evaluate_experiment6_checkpoint(
    experiment6=experiment6,
    phase1_data=eval_phase1_data,
    config=eval_config,
    random_seed=EVAL_RANDOM_SEED,
    checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
    deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
    stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
    num_eval_runs=0,
    stochastic_episode_length_limit=9999,
    save_eval_logs=EVAL_SAVE_LOGS,
    save_eval_artifacts=EVAL_SAVE_ARTIFACTS,
)


dm = full_oos_eval.deterministic_metrics or {}
pv = np.array(full_oos_eval.deterministic_portfolio)
dw = np.array(full_oos_eval.deterministic_weights)
da = np.array(full_oos_eval.deterministic_alphas)

if pv.size > 0:
    test_df_eval = getattr(full_oos_eval.env_test_deterministic, "processed_data", pd.DataFrame()).copy()
    if isinstance(test_df_eval, pd.DataFrame) and ("Date" in test_df_eval.columns):
        full_dates = (
            pd.to_datetime(test_df_eval["Date"]).dropna().drop_duplicates().sort_values().reset_index(drop=True)
        )
    else:
        full_dates = pd.Series(pd.NaT, index=np.arange(len(pv)))

    n = len(pv)
    if dw.ndim == 2 and dw.shape[0] > 0:
        n = min(n, dw.shape[0])
    if da.ndim == 2 and da.shape[0] > 0:
        n = min(n, da.shape[0])
    if len(full_dates) > 0:
        n = min(n, len(full_dates))

    full_daily_df = pd.DataFrame({
        "step": np.arange(n),
        "date": full_dates.iloc[:n].values if len(full_dates) >= n else pd.NaT,
        "portfolio_value": pv[:n],
    })
    full_daily_df["date"] = pd.to_datetime(full_daily_df["date"], errors="coerce")
    full_daily_df["daily_return"] = full_daily_df["portfolio_value"].pct_change().fillna(0.0)
    full_daily_df["cumulative_return"] = full_daily_df["portfolio_value"] / full_daily_df["portfolio_value"].iloc[0] - 1.0
    rm = full_daily_df["portfolio_value"].cummax()
    full_daily_df["drawdown"] = (full_daily_df["portfolio_value"] - rm) / rm.replace(0, np.nan)

    if dw.ndim == 2 and dw.shape[0] >= n:
        for i, t in enumerate(EVAL_ASSET_UNIVERSE):
            if i < dw.shape[1]:
                full_daily_df[f"w_{t}"] = dw[:n, i]
        if dw.shape[1] > len(EVAL_ASSET_UNIVERSE):
            full_daily_df["w_cash"] = dw[:n, -1]

    if da.ndim == 2 and da.shape[0] >= n:
        for i, t in enumerate(EVAL_ASSET_UNIVERSE):
            if i < da.shape[1]:
                full_daily_df[f"alpha_{t}"] = da[:n, i]
        if da.shape[1] > len(EVAL_ASSET_UNIVERSE):
            full_daily_df["alpha_cash"] = da[:n, -1]

    if dw.ndim == 2 and dw.shape[0] > 1:
        w2 = dw[:n]
        full_daily_df["daily_turnover"] = np.concatenate([[0.0], np.sum(np.abs(np.diff(w2, axis=0)), axis=1)])

    sharpe = dm.get("sharpe_ratio", np.nan)
    mdd_pct = dm.get("max_drawdown_abs", np.nan)
    ret_pct = dm.get("total_return", np.nan)
    mdd_txt = f"{float(mdd_pct)*100:.2f}%" if pd.notna(mdd_pct) else "n/a"
    ret_txt = f"{float(ret_pct)*100:.2f}%" if pd.notna(ret_pct) else "n/a"
    sh_txt = f"{float(sharpe):.4f}" if pd.notna(sharpe) else "n/a"

    print(
        f"[OK] Full OOS: {len(full_daily_df)} days | "
        f"Final: ${full_daily_df['portfolio_value'].iloc[-1]:,.2f} | "
        f"Sharpe: {sh_txt} | MDD: {mdd_txt} | Return: {ret_txt}"
    )
    display(full_daily_df.head())
else:
    print("[ERROR] deterministic_portfolio is empty")


## 13) Save everything + Drive backup

In [23]:
# Full OOS daily
full_csv_path = OUTPUT_DIR / "ep398_full_oos_daily.csv"
full_daily_df.to_csv(full_csv_path, index=False)

print("="*80)
print("DATA EXPORT SUMMARY")
print("="*80)
print(f"Output: {OUTPUT_DIR}")
print(f"  det_results_df:       {det_results_df.shape}")
print(f"  det_daily_data:       {len(det_daily_data)} DataFrames")
print(f"  all_stoch_results_df: {all_stoch_results_df.shape} (expected ~{EVAL_STOCH_TOTAL_RUNS_EXPECTED} rows)")
print(f"  stoch_by_horizon:     {list(stoch_by_horizon.keys())}")
print(f"  all_stoch_daily_data: {len(all_stoch_daily_data)} DataFrames")
print(f"  full_daily_df:        {full_daily_df.shape}")
print(f"  cross_horizon_df:     {cross_horizon_df.shape}")
print("="*80)



In [24]:
import zipfile

if Path("/content/drive/MyDrive").exists():
    backup_zip = Path(f"/content/drive/MyDrive/robustness_ep398_{RUN_ID}.zip")
else:
    backup_zip = OUTPUT_DIR.parent / f"robustness_ep398_{RUN_ID}.zip"

with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in OUTPUT_DIR.rglob("*"):
        if fp.is_file():
            zf.write(fp, fp.relative_to(OUTPUT_DIR))

print(f"[OK] Backup zip: {backup_zip} ({len(list(OUTPUT_DIR.rglob('*')))} files from {OUTPUT_DIR})")


---
## A1) Reload from CSV (skip if running inline)

In [ ]:
# Uncomment to reload:
# det_results_df = pd.read_csv(OUTPUT_DIR / "ep398_deterministic_summary.csv")
# all_stoch_results_df = pd.read_csv(OUTPUT_DIR / "ep398_stochastic_all_horizons.csv")
# full_daily_df = pd.read_csv(OUTPUT_DIR / "ep398_full_oos_daily.csv", parse_dates=["date"])
# cross_horizon_df = pd.read_csv(OUTPUT_DIR / "ep398_cross_horizon_summary.csv")
# stoch_by_horizon = {}
# for hl in ["1yr","2yr","3yr","4yr","full"]:
#     p = OUTPUT_DIR / f"ep398_stochastic_{hl}.csv"
#     if p.exists():
#         stoch_by_horizon[hl] = pd.read_csv(p)
# print(f"[OK] Reloaded from CSV under {OUTPUT_DIR}")
